In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WeatherPrediction")
    .getOrCreate()
)

print("Spark OK")

In [ ]:
import pandas as pd
import requests

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=48.85"
    "&longitude=2.35"
    "&start_date=2025-01-01"
    "&end_date=2025-03-31"
    "&daily=temperature_2m_mean,precipitation_sum"
    "&timezone=Europe/Paris"
)

data = requests.get(url).json()

df = pd.DataFrame({

    "date": data["daily"]["time"],

    "temperature":
        data["daily"]["temperature_2m_mean"],

    "precipitation":
        data["daily"]["precipitation_sum"]

})

df["rain_tomorrow"] = (
    df["precipitation"] > 0
).astype(int)

df.to_csv(
    "weather.csv",
    index=False
)

In [ ]:
spark_df = spark.read.csv(
    "weather.csv",
    header=True,
    inferSchema=True
)

spark_df.show()

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(

    inputCols=[
        "temperature",
        "precipitation"
    ],

    outputCol="features"
)

data = assembler.transform(
    spark_df
)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

train_data, test_data = data.randomSplit(
    [0.8, 0.2],
    seed=42
)

dt = DecisionTreeClassifier(

    featuresCol="features",

    labelCol="rain_tomorrow"
)

model = dt.fit(
    train_data
)

predictions = model.transform(
    test_data
)

predictions.select(
    "temperature",
    "precipitation",
    "prediction"
).show()